In [3]:
from rds import get_rds_connection, create_table, create_index
from dotenv import load_dotenv
load_dotenv()

conn = get_rds_connection()
create_table(conn)
create_index(conn)
conn.close()

Creating search table...
Search table created.
Creating index...
Index created.


In [1]:
from rds import get_rds_connection, backfill
from dotenv import load_dotenv
load_dotenv()

conn = get_rds_connection()
backfill(conn)
conn.commit()
conn.close()

Backfilling theorem_search_qwen8b...
Rows inserted: 9269041


In [ ]:
from rds import get_rds_connection
from dotenv import load_dotenv
load_dotenv()

conn = get_rds_connection()
cur = conn.cursor()

print("Configuring session...")

cur.execute("SET maintenance_work_mem = '8GB';")
cur.execute("SET max_parallel_maintenance_workers = 12;")
cur.execute("SET max_parallel_workers = 16;")
cur.execute("SET synchronous_commit = OFF;")

print("Creating index...")
cur.execute(r"""
CREATE INDEX CONCURRENTLY theorem_embedding_gemma_hnsw
ON theorem_embedding_gemma
USING hnsw (embedding vector_cosine_ops)
WITH (
  m = 16,
  ef_construction = 128
);
""")

print("Success.")

cur.close()
conn.close()

Configuring session...
Creating index...


In [1]:
from rds import get_rds_connection
from dotenv import load_dotenv
load_dotenv()

conn = get_rds_connection()
cur = conn.cursor()

print("Executing query...")
cur.execute(r"""
SELECT theorem_id, embedding <-> ARRAY_FILL(1.0, ARRAY[4096])::vector(4096) AS distance
FROM theorem_search_qwen8b
ORDER BY embedding <-> ARRAY_FILL(1.0, ARRAY[4096])::vector(4096)
LIMIT 20;
""")
print("Success.")

cur.close()
conn.close()

Executing query...


KeyboardInterrupt: 

In [38]:
import os
from openai import OpenAI
import numpy as np
from dotenv import load_dotenv
from itertools import combinations

load_dotenv()

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY")
)

input = ["Fermat’s last theorem", "Fermat's last theorem", "For any natural number n greater than two, the equation X to the n plus Y to the n equals Z to the n has no solutions in nonzero integers.", "The theorem is not interpretable due to corrupted or unreadable input."]

response = client.embeddings.create(
    model="Qwen/Qwen3-Embedding-8B",
    input=input
)



In [39]:
embeddings = [data.embedding for data in response.data]
binary_embeddings = [(np.array(v) > 0).astype(int) for v in embeddings]

def get_cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2)

def get_hamming_distance(v1, v2):
    return np.count_nonzero(v1 != v2)

indices = range(len(binary_embeddings))
perms = list(combinations(indices, 2))

for i, j in perms:
    dist = get_hamming_distance(binary_embeddings[i], binary_embeddings[j])
    print(f"({i+1}, {j+1}) Similarity: {get_cosine_similarity(embeddings[i], embeddings[j]):.4f}")
    print(f"       Hamming Distance: {dist} bits")
for i in range(len(input)):
    print(f"{i+1}={input[i]}")

(1, 2) Similarity: 0.9934
       Hamming Distance: 157 bits
(1, 3) Similarity: 0.7676
       Hamming Distance: 912 bits
(1, 4) Similarity: 0.6055
       Hamming Distance: 1238 bits
(2, 3) Similarity: 0.7587
       Hamming Distance: 937 bits
(2, 4) Similarity: 0.5978
       Hamming Distance: 1257 bits
(3, 4) Similarity: 0.6073
       Hamming Distance: 1204 bits
1=Fermat’s last theorem
2=Fermat's last theorem
3=For any natural number n greater than two, the equation X to the n plus Y to the n equals Z to the n has no solutions in nonzero integers.
4=The theorem is not interpretable due to corrupted or unreadable input.
